In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages.utils import trim_messages,count_tokens_approximately # used to trimmer the past messages

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI()

In [4]:
MAX_TOKENS = 150

In [5]:
def call_model(state: MessagesState):
    
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [6]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [7]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [8]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Rahul."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 10
Hi, my name is Rahul.


'Hello Rahul, nice to meet you! How can I assist you today?'

In [10]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "I am learning LangGraph."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 101
Hi, my name is Rahul.
Hello Rahul, nice to meet you! How can I assist you today?
I am learning LangGraph.
That's great to hear, LangGraph is a fascinating topic! If you have any questions or need help with anything related to LangGraph, feel free to ask me. I'll do my best to assist you.
I am learning LangGraph.


"That's great to hear! LangGraph is a fascinating field of study. If you have any specific questions or need help with any concepts, feel free to ask. I'll do my best to assist you on your learning journey."

In [11]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 141
I am learning LangGraph.
That's great to hear, LangGraph is a fascinating topic! If you have any questions or need help with anything related to LangGraph, feel free to ask me. I'll do my best to assist you.
I am learning LangGraph.
That's great to hear! LangGraph is a fascinating field of study. If you have any specific questions or need help with any concepts, feel free to ask. I'll do my best to assist you on your learning journey.
Can you explain short term memory?


"Short-term memory, also known as working memory, is a temporary storage system that holds a small amount of information for a short period of time, usually for a few seconds to a minute. This type of memory allows us to hold onto information while we are actively processing it or using it in some way.\n\nShort-term memory is involved in tasks such as remembering a phone number long enough to dial it, following directions, or holding a person's name in mind while engaged in a conversation. It is limited in capacity and duration, meaning that it can only hold a small amount of information at a time and that information will fade away if not actively rehearsed or transferred to long-term memory through encoding processes.\n\nShort-term memory is an important component of overall cognitive function and plays a crucial role in many aspects of daily life, such as learning, problem-solving, decision-making, and reasoning. It is closely linked to attention and concentration, as well as to oth

In [12]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
What is my name?


"I'm sorry, I am a language model AI and do not have access to your personal information."

In [13]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Rahul.
------------------------------------------------------------------------------------------------------------------------
Hello Rahul, nice to meet you! How can I assist you today?
------------------------------------------------------------------------------------------------------------------------
I am learning LangGraph.
------------------------------------------------------------------------------------------------------------------------
That's great to hear, LangGraph is a fascinating topic! If you have any questions or need help with anything related to LangGraph, feel free to ask me. I'll do my best to assist you.
------------------------------------------------------------------------------------------------------------------------
I am learning LangGraph.
------------------------------------------------------------------------------------------------------------------------
That's great to hear! LangGraph is a fascinating field of study. If you have any 